# ChurnSignal — Milestone 4: Baseline Model (Logistic Regression)
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 4**! Having engineered our dynamic velocity features and established strict Out-of-Time partitions, we now train our **very first machine learning models**.

---

### Why Start with a Simple Baseline?
> *"Before you build a Ferrari, build a bicycle. You need to know that the road goes where you think it does."*

In professional ML engineering, a baseline serves three critical purposes:
1. **Performance Anchor**: Establishes the minimum standard that any complex model (like LightGBM or XGBoost) must beat to justify its complexity.
2. **Infrastructure End-to-End Test**: Verifies that data loading, feature pipelines, splitting, and scoring operate without bugs or data leakage.
3. **Transparent Audit**: Linear model coefficients allow us to inspect whether the model has learned sensible domain relationships.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_prediction_eligible_data
from src.features.engineer import engineer_all_features
from src.features.pipeline import chronological_split, ChurnFeaturePipeline
from src.models.baselines import (
    train_dummy_baseline,
    train_logistic_baseline,
    get_logistic_coefficients,
    save_model
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Libraries and modules loaded.')

## 1. Load Data, Engineer Features & Partition
We load the 74,817 eligible customer-month records, calculate velocity and rolling features, and execute our Out-of-Time split.


In [ ]:
raw_df = load_prediction_eligible_data()
engineered_df = engineer_all_features(raw_df)
train_df, val_df, test_df = chronological_split(engineered_df)

pipeline = ChurnFeaturePipeline()
X_train = pipeline.fit_transform(train_df)
X_val = pipeline.transform(val_df)
X_test = pipeline.transform(test_df)

y_train = train_df['churned_next_month'].values
y_val = val_df['churned_next_month'].values
y_test = test_df['churned_next_month'].values

print(f'Train Matrix Shape : {X_train.shape} | Positive Targets: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Val Matrix Shape   : {X_val.shape} | Positive Targets: {y_val.sum():,} ({y_val.mean()*100:.2f}%)')
print(f'Test Matrix Shape  : {X_test.shape} | Positive Targets: {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

## 2. Baseline 1: The Dummy Classifier (The Accuracy Trap)

### Concept Intuition:
A `DummyClassifier(strategy='most_frequent')` simply predicts `0` (retained) for every single account without looking at any features.

Notice that it scores **98.34% accuracy** on the validation set, but its **Recall is 0.00%** and **PR-AUC is 0.0166** (random chance). It is 100% useless for business retention.


In [ ]:
dummy_model = train_dummy_baseline(X_train, y_train)
dummy_preds = dummy_model.predict(X_val)

print(f'Dummy Accuracy : {accuracy_score(y_val, dummy_preds)*100:.2f}%')
print(f'Dummy Recall   : {recall_score(y_val, dummy_preds, zero_division=0):.4f}')
print(f'Dummy PR-AUC   : {average_precision_score(y_val, dummy_model.predict_proba(X_val)[:, 1]):.4f}')

## 3. Baseline 2: Standard Logistic Regression (Unweighted)

### Concept Intuition:
Logistic Regression calculates a linear score $z = \mathbf{w}^T \mathbf{x} + b$ and maps it to a probability using the sigmoid function:
$$P(y=1) = \frac{1}{1 + e^{-z}}$$

With standard loss minimization, the model optimizes overall error. Because 98.6% of samples are 0, the model tends to predict small probabilities close to 0 for almost everyone.


In [ ]:
lr_standard = train_logistic_baseline(X_train, y_train, class_weight=None)
lr_std_probs = lr_standard.predict_proba(X_val)[:, 1]

print(f'Standard LR - ROC-AUC : {roc_auc_score(y_val, lr_std_probs):.4f}')
print(f'Standard LR - PR-AUC  : {average_precision_score(y_val, lr_std_probs):.4f}')

## 4. Baseline 3: Balanced Logistic Regression (`class_weight='balanced'`)

### Concept Intuition:
Under `class_weight='balanced'`, Scikit-Learn penalizes misclassifying a rare churner roughly **75x more heavily** than misclassifying a non-churner:
$$w_j = \frac{N}{2 \cdot N_j}$$

This forces the optimization algorithm to shift its decision boundary and pay serious attention to rare churn warning signs.


In [ ]:
lr_balanced = train_logistic_baseline(X_train, y_train, class_weight='balanced')
lr_bal_probs = lr_balanced.predict_proba(X_val)[:, 1]

print(f'Balanced LR - ROC-AUC : {roc_auc_score(y_val, lr_bal_probs):.4f}')
print(f'Balanced LR - PR-AUC  : {average_precision_score(y_val, lr_bal_probs):.4f}')

## 5. Auditing Model Weights: What Did the Baseline Learn?

### Odds Ratio Intuition:
- **Coefficient > 0 (Odds Ratio > 1.0)**: Feature **increases** churn risk.
  - E.g. `product_incident`, `payment_failures`, `support_tickets`.
- **Coefficient < 0 (Odds Ratio < 1.0)**: Feature **protects against** churn.
  - E.g. `rolling_mean_usage_3m`, `plan_type_Enterprise`, `active_seats`.


In [ ]:
feature_names = pipeline.get_feature_names()
coef_table = get_logistic_coefficients(lr_balanced, feature_names)

print('--- Top 7 Features INCREASING Churn Risk ---')
display(coef_table.head(7))

print('\n--- Top 7 Features PROTECTING Against Churn ---')
display(coef_table.tail(7).iloc[::-1])

## 6. Predicted Risk Score Distribution

Let's inspect how the model scores retained accounts vs. churning accounts on the validation set.


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(lr_bal_probs[y_val == 0], label='Retained (0)', fill=True, color='#4C72B0', alpha=0.5)
sns.kdeplot(lr_bal_probs[y_val == 1], label='Churn Next Month (1)', fill=True, color='#C44E52', alpha=0.5)
plt.title('Validation Set: Predicted Churn Probability Distribution (Balanced LR)', fontweight='bold')
plt.xlabel('Predicted Churn Probability')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Serialize Baseline Model Artifact
We save our trained model and preprocessing pipeline to `artifacts/` for reproducible inference.


In [ ]:
save_model(lr_balanced, 'baseline_logistic_regression.joblib')
save_model(pipeline, 'feature_pipeline.joblib')
print('Models and feature pipelines successfully serialized to artifacts/.')

## 8. Milestone 4 Summary & Baseline Benchmark Table

| Model | Weighting | Validation ROC-AUC | Validation PR-AUC | Key Advantage |
|---|---|---|---|---|
| **Dummy Classifier** | N/A | 0.5000 | 0.0166 | Null benchmark |
| **Standard Logistic Regression** | None | ~0.8027 | ~0.0699 | Fast, simple linear baseline |
| **Balanced Logistic Regression** | Balanced | **~0.8070** | **~0.0712** | Penalizes rare-event errors |

### Next Step: Milestone 5 — Imbalance-Aware Evaluation
Now that we have probability scores, how do we choose the optimal decision threshold? How do we balance false alarms vs missed churners? We will dive deep into Precision, Recall, and PR curves!
